In [ ]:
#Install Dependencies
!pip install -q groq fpdf2 openpyxl requests pandas pillow reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.0 MB/s eta 0:00:00


In [ ]:
# Upload Dataset Files
from google.colab import files
print('📂 Upload: swiggy.csv, HotelData.csv, Data_Train.xlsx')
uploaded = files.upload()
print('✅ Uploaded:', list(uploaded.keys()))

📂 Upload: swiggy.csv, HotelData.csv, Data_Train.xlsx


Saving Data_Train.xlsx to Data_Train.xlsx
Saving HotelData.csv to HotelData.csv
Saving swiggy.csv to swiggy.csv
✅ Uploaded: ['Data_Train.xlsx', 'HotelData.csv', 'swiggy.csv']


In [ ]:
# ══ CELL 3: API Keys ══════════════════════════════════
from getpass import getpass
GROQ_API_KEY        = getpass('🔑 Enter your GROQ API Key: ').strip()
OPENTRIPMAP_API_KEY = getpass('🗺️  Enter your OpenTripMap API Key): ').strip()

🔑 Enter your GROQ API Key: ··········
🗺️  Enter your OpenTripMap API Key): ··········


In [ ]:
# ── CELL 4: Imports & Load + CLEAN datasets ────────────
import os, ast, re, warnings
import requests, urllib.parse
import pandas as pd
import numpy as np
from groq import Groq
from IPython.display import display, FileLink
warnings.filterwarnings('ignore')

groq_client = Groq(api_key=GROQ_API_KEY)

# ── Load ───────────────────────────────────────────────
swiggy_df = pd.read_csv('swiggy.csv')
hotel_df  = pd.read_csv('HotelData.csv')
flight_df = pd.read_excel('Data_Train.xlsx')

# ══════════════════════════════════════════════════════
# FLIGHT CLEANING
# Cols: Airline, Source, Destination, Price (int), Total_Stops
# ══════════════════════════════════════════════════════
flight_df.dropna(subset=['Source', 'Destination', 'Price'], inplace=True)
flight_df['Price'] = pd.to_numeric(flight_df['Price'], errors='coerce')
flight_df.dropna(subset=['Price'], inplace=True)
flight_df['Price'] = flight_df['Price'].astype(int)
flight_df['Source']      = flight_df['Source'].str.strip().replace({'Banglore': 'Bangalore'})
flight_df['Destination'] = flight_df['Destination'].str.strip().replace(
    {'Banglore': 'Bangalore', 'New Delhi': 'Delhi'})
flight_df['Total_Stops'] = flight_df['Total_Stops'].fillna('unknown')

# ══════════════════════════════════════════════════════
# HOTEL CLEANING
# Cols: Name→hotel_name, City, Price (float), Rating (float), Address
# ══════════════════════════════════════════════════════
hotel_df.dropna(subset=['Name', 'City', 'Price'], inplace=True)
hotel_df['Price']  = pd.to_numeric(hotel_df['Price'],  errors='coerce')
hotel_df['Rating'] = pd.to_numeric(hotel_df['Rating'], errors='coerce')
hotel_df.dropna(subset=['Price'], inplace=True)
hotel_df = hotel_df[hotel_df['Price'] > 0]
hotel_df['City'] = hotel_df['City'].str.strip()
hotel_df['Name'] = hotel_df['Name'].str.strip()
hotel_df.rename(columns={'Name': 'hotel_name'}, inplace=True)

# ══════════════════════════════════════════════════════
# SWIGGY CLEANING
# Cols: name, city→city_clean, cost→cost_numeric, rating→rating_clean, cuisine
# city values: 'Locality,CityName' — extract last part as base city
# cost values: '₹ 200' — strip symbol, convert to float
# rating: '--' means no rating
# ══════════════════════════════════════════════════════
swiggy_df['city_clean'] = swiggy_df['city'].str.split(',').str[-1].str.strip()
swiggy_df['name']       = swiggy_df['name'].str.strip()
swiggy_df['cost_numeric'] = (
    swiggy_df['cost'].astype(str)
    .str.replace(r'[\u20B9\u20b9Rs.\s,]', '', regex=True)
    .replace('', np.nan)
)
swiggy_df['cost_numeric']  = pd.to_numeric(swiggy_df['cost_numeric'],  errors='coerce')
swiggy_df['rating_clean']  = swiggy_df['rating'].replace('--', np.nan)
swiggy_df['rating_clean']  = pd.to_numeric(swiggy_df['rating_clean'], errors='coerce')
swiggy_df.dropna(subset=['name', 'city_clean'], inplace=True)

print('✅ Datasets loaded and cleaned')
print(f'  Swiggy  : {len(swiggy_df):,} restaurants')
print(f'  Hotels  : {len(hotel_df):,} hotels')
print(f'  Flights : {len(flight_df):,} flights')
print(f'\n  Available flight sources      : {", ".join(flight_df["Source"].unique())}')
print(f'  Available flight destinations : {", ".join(flight_df["Destination"].unique())}')
print(f'  Hotel cities (sample)         : {", ".join(hotel_df["City"].unique()[:10])}')

✅ Datasets loaded and cleaned
  Swiggy  : 148,455 restaurants
  Hotels  : 11,813 hotels
  Flights : 10,683 flights

  Available flight sources      : Bangalore, Kolkata, Delhi, Chennai, Mumbai
  Available flight destinations : Delhi, Bangalore, Cochin, Kolkata, Hyderabad
  Hotel cities (sample)         : Mahabaleshwar, Mahabalipuram, Manali, Mysore, Mussoorie, Mathura, Mcleodganj, Nainital, Munnar, Mumbai


In [ ]:
# ══ CELL 4: Imports & Clean Datasets ══════════════════
import os, ast, re, requests, warnings
import pandas as pd
import numpy as np
import urllib.parse
from fpdf import FPDF
from groq import Groq
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

groq_client = Groq(api_key=GROQ_API_KEY)

# ── Load ───────────────────────────────────────────────
swiggy_df = pd.read_csv('swiggy.csv')
hotel_df  = pd.read_csv('HotelData.csv')
flight_df = pd.read_excel('Data_Train.xlsx')

# ── FLIGHT cleaning ────────────────────────────────────
flight_df.dropna(subset=['Source','Destination','Price'], inplace=True)
flight_df['Price'] = pd.to_numeric(flight_df['Price'], errors='coerce')
flight_df.dropna(subset=['Price'], inplace=True)
flight_df['Price'] = flight_df['Price'].astype(int)
flight_df['Source']      = flight_df['Source'].str.strip().replace({'Banglore':'Bangalore'})
flight_df['Destination'] = flight_df['Destination'].str.strip().replace({'Banglore':'Bangalore','New Delhi':'Delhi'})
flight_df['Total_Stops'] = flight_df['Total_Stops'].fillna('unknown')

# ── HOTEL cleaning ─────────────────────────────────────
hotel_df.dropna(subset=['Name','City','Price'], inplace=True)
hotel_df['Price']  = pd.to_numeric(hotel_df['Price'],  errors='coerce')
hotel_df['Rating'] = pd.to_numeric(hotel_df['Rating'], errors='coerce')
hotel_df.dropna(subset=['Price'], inplace=True)
hotel_df = hotel_df[hotel_df['Price'] > 0].copy()
hotel_df['City'] = hotel_df['City'].str.strip()
hotel_df.rename(columns={'Name':'hotel_name'}, inplace=True)

# ── SWIGGY cleaning ────────────────────────────────────
swiggy_df['city_clean']   = swiggy_df['city'].str.split(',').str[-1].str.strip()
swiggy_df['name']         = swiggy_df['name'].str.strip()
swiggy_df['cost_numeric'] = (
    swiggy_df['cost'].astype(str)
    .str.replace(r'[Rs.INR,\s]', '', regex=True)
    .str.replace(u'\u20b9','', regex=False)
    .replace('',np.nan)
)
swiggy_df['cost_numeric'] = pd.to_numeric(swiggy_df['cost_numeric'], errors='coerce')
swiggy_df['rating_clean'] = pd.to_numeric(swiggy_df['rating'].replace('--',np.nan), errors='coerce')
swiggy_df.dropna(subset=['name','city_clean'], inplace=True)

print('✅ All datasets loaded & cleaned')
print(f'   Swiggy : {len(swiggy_df):,} rows')
print(f'   Hotels : {len(hotel_df):,} rows')
print(f'   Flights: {len(flight_df):,} rows')

✅ All datasets loaded & cleaned
   Swiggy : 148,455 rows
   Hotels : 11,813 rows
   Flights: 10,683 rows


In [ ]:
# ══ CELL 5: Helper Functions ═══════════════════════════
def enc(q):   return urllib.parse.quote(str(q))
def cln(s):   return str(s).strip().lower()
def gmaps(place, city=''):
    q = (place + ' ' + city + ' India').strip()
    return f'https://www.google.com/maps/search/?api=1&query={enc(q)}'

# ── Weather ────────────────────────────────────────────
def get_weather(city):
    try:
        r = requests.get(f'https://wttr.in/{enc(city)}?format=j1', timeout=6).json()
        d = r['weather'][0]
        return {
            'temp_c'  : d['avgtempC'],
            'desc'    : d['hourly'][4]['weatherDesc'][0]['value'],
            'humidity': d['hourly'][4]['humidity'],
            'wind'    : d['hourly'][4]['windspeedKmph']
        }
    except:
        return {'temp_c':'N/A','desc':'N/A','humidity':'N/A','wind':'N/A'}

# ── Wikipedia snippet ──────────────────────────────────
def wiki_snippet(name):
    try:
        r = requests.get(
            f'https://en.wikipedia.org/api/rest_v1/page/summary/{enc(name.split("(")[0].strip())}',
            timeout=4).json()
        return r.get('extract','')[:180]
    except:
        return ''

# ── Junk filter ────────────────────────────────────────
JUNK = ['community','network','node','boundary','residential',
        'administrative','unknown','unnamed','possible','area',
        'district','region','zone','ward','quarter',
        'hospital','crematorium','cemetery','graveyard','school','college',
        'smashana','theater','cinema','pvr','multiplex']

def is_valid(name):
    return (isinstance(name, str) and len(name) > 3
            and not any(j in name.lower() for j in JUNK))
# ── OTM valid kind strings per preference ─────────────
# These are the EXACT OTM taxonomy strings (verified against OTM docs)
OTM_KINDS = {
    'cultural'  : 'historic,cultural,architecture,religion,museums',
    'adventure' : 'natural,amusements,sport,water',
    'foodie'    : 'foods,restaurants,markets,fast_food',
    'relaxation': 'natural,gardens,parks,beaches,thermal_springs',
}

# ── Geocode city using Nominatim (more reliable than OTM geoname) ──
def geocode_city(city):
    """
    Use Nominatim (OpenStreetMap) to get accurate lat/lon for any Indian city.
    Strictly restricts to India and returns city-level coordinates only.
    """
    try:
        url = (
            f'https://nominatim.openstreetmap.org/search'
            f'?q={enc(city)}+India'
            f'&countrycodes=IN'
            f'&limit=1'
            f'&format=json'
            f'&addressdetails=1'
        )
        headers = {'User-Agent': 'TravelPlannerPro/1.0'}
        r = requests.get(url, headers=headers, timeout=7).json()
        if r:
            return float(r[0]['lat']), float(r[0]['lon'])
    except Exception as e:
        print(f'  Geocode error: {e}')
    return None, None

# ── OTM place fetcher with strict city-boundary filter ──
def get_places_otm(city, preference='cultural', limit=12):
    """
    Fetch tourist attractions strictly within the given city using OTM.
    Falls back to LLM if OTM fails or returns too few results.
    Uses Nominatim for reliable geocoding instead of OTM geoname.
    """
    kinds = OTM_KINDS.get(preference, 'historic,cultural,architecture')
    city_clean = city.strip()

    # Step 1: Geocode with Nominatim (much more reliable for Indian cities)
    lat, lon = geocode_city(city_clean)
    if not lat or not lon:
        # Fallback: try OTM's own geoname endpoint
        try:
            geo = requests.get(
                f'https://api.opentripmap.com/0.1/en/places/geoname'
                f'?name={enc(city_clean)}&country=IN&apikey={OPENTRIPMAP_API_KEY}',
                timeout=6).json()
            lat = geo.get('lat')
            lon = geo.get('lon')
        except:
            pass
    if not lat or not lon:
        print(f'  Could not geocode {city_clean} — switching to LLM.')
        return [], None, None

    print(f'  Geocoded {city_clean}: lat={lat:.4f}, lon={lon:.4f}')

    # Step 2: Fetch attractions from OTM with a tight radius (8 km = city core)
    # Use rate=1 minimum to avoid junk POIs
    places_out, seen = [], set()
    for kind_group in kinds.split(','):
        kind_group = kind_group.strip()
        if not kind_group:
            continue
        try:
            url = (
                f'https://api.opentripmap.com/0.1/en/places/radius'
                f'?radius=8000'          # 8 km — keeps results inside the city
                f'&lon={lon}&lat={lat}'
                f'&kinds={kind_group}'   # one kind at a time avoids 400 errors
                f'&rate=1'
                f'&format=json'
                f'&limit=20'
                f'&apikey={OPENTRIPMAP_API_KEY}'
            )
            raw = requests.get(url, timeout=8).json()
            if isinstance(raw, dict) and raw.get('error'):
                print(f'  OTM kind "{kind_group}" error: {raw["error"]} — skipping')
                continue
            if not isinstance(raw, list):
                continue
            for item in raw:
                nm = str(item.get('name', '')).strip()
                if nm and nm not in seen and is_valid(nm):
                    seen.add(nm)
                    places_out.append({
                        'name' : nm,
                        'gmaps': gmaps(nm, city_clean),
                        'wiki' : wiki_snippet(nm)
                    })
        except Exception as e:
            print(f'  OTM kind "{kind_group}" request failed: {e}')
            continue

    print(f'  OTM returned {len(places_out)} valid places for {city_clean}')
    return places_out[:limit], lat, lon

# ── LLM fallback for places ────────────────────────────
def get_places_llm(city, preference='cultural'):
    """
    Ask LLM for real, verifiable attractions strictly inside the given city.
    Uses a strict prompt to avoid hallucinations and nearby-city confusion.
    """
    pref_desc = {
        'cultural'  : 'historical monuments, temples, museums, heritage sites',
        'adventure' : 'trekking trails, nature spots, waterfalls, adventure parks',
        'foodie'    : 'famous food streets, local markets, iconic eateries',
        'relaxation': 'scenic viewpoints, gardens, lakes, peaceful retreats',
    }.get(preference, 'popular tourist spots')

    prompt = (
        f'List exactly 10 real, well-known tourist attractions in {city}, India '
        f'that are suitable for a {preference} traveller ({pref_desc}). '
        f'STRICT RULES: '
        f'(1) Every place MUST be physically located INSIDE {city} — do NOT include places from nearby cities or districts. '
        f'(2) Only include places that genuinely exist and are verifiable. '
        f'(3) Output ONLY a Python list of strings, nothing else. '
        f'Format: ["Place1","Place2","Place3","Place4","Place5","Place6","Place7","Place8","Place9","Place10"]'
    )
    try:
        res = groq_client.chat.completions.create(
            model='llama3-70b-8192',
            messages=[
                {'role': 'system',
                 'content': (
                     'You are a strict travel data assistant. '
                     'Output ONLY a valid Python list of strings. '
                     'Every place must be PHYSICALLY LOCATED inside the exact city named. '
                     'NEVER include places from nearby towns or districts. '
                     'NEVER invent place names.')},
                {'role': 'user', 'content': prompt}
            ], temperature=0)
        raw = res.choices[0].message.content.strip()
        # Robust extraction of the list
        start = raw.find('[')
        end   = raw.rfind(']') + 1
        if start == -1 or end == 0:
            raise ValueError('No list found in LLM response')
        lst = ast.literal_eval(raw[start:end])
        if not isinstance(lst, list):
            raise ValueError('LLM did not return a list')
        results = []
        for n in lst:
            n = str(n).strip()
            if is_valid(n):
                results.append({
                    'name' : n,
                    'gmaps': gmaps(n, city),
                    'wiki' : wiki_snippet(n)
                })
        return results
    except Exception as e:
        print(f'  LLM places error: {e}')
        return []

# ── LLM: suggest restaurants with gmaps links ──────────
def get_restaurants_llm(city, preference='cultural', n=8):
    """Get real restaurant names via LLM when city not in Swiggy dataset."""
    pref_food = {
        'foodie'    : 'famous local delicacies, street food, must-try restaurants',
        'cultural'  : 'traditional authentic local cuisine restaurants',
        'adventure' : 'casual cafes, dhaba-style, energizing food spots',
        'relaxation': 'rooftop restaurants, scenic dining, peaceful cafes'
    }
    food_hint = pref_food.get(preference, 'popular local restaurants')
    try:
        res = groq_client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
              {'role':'system',
               'content':(
                   'You are a food travel expert. '
                   'Output ONLY a valid Python list of dicts. '
                   'List ONLY real, verifiable restaurant names that actually exist. '
                   'NEVER invent names. '
                   'Format exactly: [{"name":"X","cuisine":"Y","specialty":"Z"},...] — nothing else.')},
              {'role':'user',
               'content':(
                   f'List {n} real well-known restaurants in {city}, India '
                   f'focusing on {food_hint}. '
                   f'Only places that genuinely exist. '
                   f'Output ONLY the Python list.')}
            ], temperature=0)
        raw = res.choices[0].message.content.strip()
        lst = ast.literal_eval(raw[raw.find('['):raw.rfind(']')+1])
        return [{
            'name'     : r.get('name',''),
            'cuisine'  : r.get('cuisine',''),
            'specialty': r.get('specialty',''),
            'gmaps'    : gmaps(r.get('name',''), city),
            'source'   : 'LLM Suggested'
        } for r in lst if r.get('name','').strip()]
    except:
        return []

print('✅ Helper functions ready')


✅ Helper functions ready


In [ ]:
# ══ CELL 6: User Inputs ════════════════════════════════
print('='*55)
print('  TRAVEL PLANNER PRO — Fill in your trip details')
print('='*55)

departure   = input('\n✈️  Departure City          : ').strip()
destination = input('🏁 Destination City        : ').strip()
travelers   = int(input('👥 Number of Travelers     : '))
start_date  = input('📅 Start Date (DD-MM-YYYY) : ').strip()
end_date    = input('📅 End Date   (DD-MM-YYYY) : ').strip()
budget      = float(input('💰 Total Budget (Rs.)      : '))

print('\n👤 Traveler Profiles (enter for each traveler):')
traveler_profiles = []
for i in range(travelers):
    print(f'\n  --- Traveler {i+1} ---')
    age    = input(f'  Age    : ').strip()
    gender = input(f'  Gender (Male/Female/Other): ').strip()
    style  = input(f'  Style  (Casual/Streetwear/Formal/Bohemian/Minimalist/Sporty/Traditional): ').strip()
    traveler_profiles.append({'age':age,'gender':gender,'style':style})

print('\n🎯 Trip Preference:')
print('   1. Foodie      — Local food, street eats, restaurants')
print('   2. Cultural    — History, temples, museums, heritage')
print('   3. Adventure   — Trekking, sports, thrills, nature')
print('   4. Relaxation  — Scenic spots, cafes, peaceful parks')
pref_choice = input('\nEnter preference (1/2/3/4): ').strip()
pref_map    = {'1':'foodie','2':'cultural','3':'adventure','4':'relaxation'}
preference  = pref_map.get(pref_choice, 'cultural')

from datetime import datetime
sd_obj   = datetime.strptime(start_date, '%d-%m-%Y').date()
ed_obj   = datetime.strptime(end_date, '%d-%m-%Y').date()
days = max((ed_obj - sd_obj).days, 1)

print(f'\n✅ Planning {days}-day {preference.upper()} trip')
print(f'   {departure} → {destination} | {travelers} traveler(s) | Budget Rs.{budget:,.0f}')

  TRAVEL PLANNER PRO — Fill in your trip details

✈️  Departure City          : Mumbai
🏁 Destination City        : Mysore
👥 Number of Travelers     : 2
📅 Start Date (DD-MM-YYYY) : 04-04-2026
📅 End Date   (DD-MM-YYYY) : 07-04-2026
💰 Total Budget (Rs.)      : 60000

👤 Traveler Profiles (enter for each traveler):

  --- Traveler 1 ---
  Age    : 21
  Gender (Male/Female/Other): Male
  Style  (Casual/Streetwear/Formal/Bohemian/Minimalist/Sporty/Traditional): Casual

  --- Traveler 2 ---
  Age    : 21
  Gender (Male/Female/Other): Female
  Style  (Casual/Streetwear/Formal/Bohemian/Minimalist/Sporty/Traditional): Traditional

🎯 Trip Preference:
   1. Foodie      — Local food, street eats, restaurants
   2. Cultural    — History, temples, museums, heritage
   3. Adventure   — Trekking, sports, thrills, nature
   4. Relaxation  — Scenic spots, cafes, peaceful parks

Enter preference (1/2/3/4): 2

✅ Planning 3-day CULTURAL trip
   Mumbai → Mysore | 2 traveler(s) | Budget Rs.60,000


In [ ]:
# ══ CELL 7: Filter Datasets ════════════════════════════
hotels  = hotel_df[hotel_df['City'].apply(cln).str.contains(cln(destination), na=False)]
foods   = swiggy_df[swiggy_df['city_clean'].apply(cln).str.contains(cln(destination), na=False)]
flights = flight_df[
    flight_df['Source'].apply(cln).str.contains(cln(departure)) &
    flight_df['Destination'].apply(cln).str.contains(cln(destination))
]

print(f'Dataset matches for {destination}:')
print(f'  Hotels     : {len(hotels)}')
print(f'  Restaurants: {len(foods)}')
print(f'  Flights    : {len(flights)}')
if len(hotels)==0:
    print('  ⚠️  No hotels in dataset — will estimate via LLM')
if len(foods)==0:
    print('  ⚠️  No restaurants in dataset — will fetch real ones via LLM')
if len(flights)==0:
    print('  ⚠️  No flights in dataset — will estimate')

Dataset matches for Mysore:
  Hotels     : 249
  Restaurants: 656
  Flights    : 0
  ⚠️  No flights in dataset — will estimate


In [ ]:
# ══ CELL 8: Live Weather ═══════════════════════════════
print('\nFetching live weather...')
weather = get_weather(destination)
print(f'\n🌤️  {destination} Weather')
print(f'   {weather["desc"]} | {weather["temp_c"]}°C | '
      f'Humidity {weather["humidity"]}% | Wind {weather["wind"]} km/h')


Fetching live weather...

🌤️  Mysore Weather
   Partly Cloudy  | 27°C | Humidity 30% | Wind 14 km/h


In [ ]:
# ══ CELL 9: Real Attractions via OTM (with LLM fallback) ══
print(f'\nFetching real attractions strictly inside {destination}...')
print('=' * 60)

places      = []
city_lat    = None
city_lon    = None
otm_success = False

# ── Step 1: Try OTM ───────────────────────────────────
try:
    places, city_lat, city_lon = get_places_otm(destination, preference)
    if len(places) >= 3:
        otm_success = True
        print(f'  ✅ OTM: {len(places)} places found in {destination}')
    else:
        print(f'  ⚠️  OTM returned only {len(places)} results — switching to LLM fallback...')
except Exception as e:
    print(f'  ⚠️  OTM failed: {e} — switching to LLM fallback...')

# ── Step 2: LLM fallback if OTM insufficient ──────────
if not otm_success:
    print(f'  🧠 Asking LLM for real {preference} places strictly inside {destination}...')
    llm_places = get_places_llm(destination, preference)

    if llm_places:
        # Merge: keep any OTM results + add LLM ones (deduplicated)
        existing_names = {p['name'].lower() for p in places}
        for p in llm_places:
            if p['name'].lower() not in existing_names:
                places.append(p)
                existing_names.add(p['name'].lower())
        print(f'  ✅ LLM fallback: total {len(places)} places for {destination}')
    else:
        print(f'  ⚠️  LLM also returned no results. Using city landmark as placeholder.')
        places = [{
            'name' : f'{destination} City Centre',
            'gmaps': gmaps(destination, destination),
            'wiki' : f'The main hub of {destination}, India.'
        }]

# ── Step 3: Build clean name list for downstream cells ─
clean_place_names = [p['name'] for p in places if is_valid(p.get('name', ''))]
if not clean_place_names:
    clean_place_names = [f'{destination} city centre']

# ── Step 4: Display results ────────────────────────────
source_label = '✅ OpenTripMap' if otm_success else '🧠 LLM (OTM unavailable)'
print(f'\n🗺️  PLACES TO VISIT ({preference.title()} focus) — {destination}')
print(f'   Source: {source_label}')
print('=' * 60)

for i, p in enumerate(places, 1):
    name = p.get('name', 'Unknown')
    print(f'\n  {i:2d}. 📍 {name}')
    wiki = p.get('wiki', '')
    if wiki:
        # Wrap long descriptions neatly
        desc = wiki[:140].rstrip()
        if len(wiki) > 140:
            desc += '...'
        print(f'      {desc}')
    maps_link = p.get('gmaps', '#')
    print(f'      🔗 {maps_link}')

print(f'\n✅ {len(places)} places ready for itinerary generation.')



Fetching real attractions strictly inside Mysore...
  Geocoded Mysore: lat=12.3052, lon=76.6554
  OTM returned 48 valid places for Mysore
  ✅ OTM: 12 places found in Mysore

🗺️  PLACES TO VISIT (Cultural focus) — Mysore
   Source: ✅ OpenTripMap

   1. 📍 Narasimharaja Wodeyar
      🔗 https://www.google.com/maps/search/?api=1&query=Narasimharaja%20Wodeyar%20Mysore%20India

   2. 📍 Krishnaraja Wodeyar
      🔗 https://www.google.com/maps/search/?api=1&query=Krishnaraja%20Wodeyar%20Mysore%20India

   3. 📍 Mahishasura
      🔗 https://www.google.com/maps/search/?api=1&query=Mahishasura%20Mysore%20India

   4. 📍 Opera Theatre
      🔗 https://www.google.com/maps/search/?api=1&query=Opera%20Theatre%20Mysore%20India

   5. 📍 Woodlands
      🔗 https://www.google.com/maps/search/?api=1&query=Woodlands%20Mysore%20India

   6. 📍 Ranjith
      🔗 https://www.google.com/maps/search/?api=1&query=Ranjith%20Mysore%20India

   7. 📍 Lido
      🔗 https://www.google.com/maps/search/?api=1&query=Lido%20Mysore%

In [ ]:
# ══ CELL 10: Restaurants (Dataset or LLM) ═════════════
print(f'\n🍽️  RESTAURANTS — {destination}')
print('='*60)

restaurant_data = []   # list of dicts with name/cuisine/cost/gmaps/source
food_cost_val   = None

if not foods.empty:
    top_foods = foods.sort_values('rating_clean', ascending=False, na_position='last').head(8)
    for _, row in top_foods.iterrows():
        restaurant_data.append({
            'name'    : str(row['name']),
            'cuisine' : str(row.get('cuisine','')) if pd.notna(row.get('cuisine')) else '',
            'cost'    : row['cost_numeric'],
            'rating'  : row['rating_clean'],
            'gmaps'   : gmaps(str(row['name']), destination),
            'source'  : '✅ Dataset'
        })
    avg_cost = foods['cost_numeric'].dropna()
    if not avg_cost.empty:
        food_cost_val = round(avg_cost.mean() * 3 * days * travelers, 2)
else:
    print(f'  No Swiggy data — fetching real restaurants via LLM...')
    llm_rests = get_restaurants_llm(destination, preference)
    for r in llm_rests:
        restaurant_data.append({
            'name'    : r['name'],
            'cuisine' : r['cuisine'],
            'cost'    : None,
            'rating'  : None,
            'gmaps'   : r['gmaps'],
            'source'  : '⚠️ LLM Suggested'
        })

for r in restaurant_data:
    print(f"\n  🍴 {r['name']}  [{r['source']}]")
    if r['cuisine']: print(f"      Cuisine : {r['cuisine']}")
    if pd.notna(r.get('rating')) and r['rating']: print(f"      Rating  : ⭐ {r['rating']:.1f}")
    if pd.notna(r.get('cost'))   and r['cost']  : print(f"      ~Cost   : Rs.{r['cost']:.0f}/person")
    print(f"      Maps    : {r['gmaps']}")

restaurant_list = [r['name'] for r in restaurant_data]


🍽️  RESTAURANTS — Mysore

  🍴 The Cafe Buono  [✅ Dataset]
      Cuisine : Italian,Pizzas
      Rating  : ⭐ 4.8
      ~Cost   : Rs.300/person
      Maps    : https://www.google.com/maps/search/?api=1&query=The%20Cafe%20Buono%20Mysore%20India

  🍴 Greendays  [✅ Dataset]
      Cuisine : Healthy Food,Salads
      Rating  : ⭐ 4.8
      ~Cost   : Rs.500/person
      Maps    : https://www.google.com/maps/search/?api=1&query=Greendays%20Mysore%20India

  🍴 Bombat Buns  [✅ Dataset]
      Cuisine : Bakery,Beverages
      Rating  : ⭐ 4.7
      ~Cost   : Rs.300/person
      Maps    : https://www.google.com/maps/search/?api=1&query=Bombat%20Buns%20Mysore%20India

  🍴 Grameen Kulfi  [✅ Dataset]
      Cuisine : Ice Cream,Desserts
      Rating  : ⭐ 4.7
      ~Cost   : Rs.120/person
      Maps    : https://www.google.com/maps/search/?api=1&query=Grameen%20Kulfi%20Mysore%20India

  🍴 NIC Natural Ice Creams  [✅ Dataset]
      Cuisine : Desserts,Ice Cream
      Rating  : ⭐ 4.6
      ~Cost   : Rs.120/pers

In [ ]:
# ══ CELL 11: Flights ══════════════════════════════════
print(f'\n✈️  FLIGHTS — {departure} → {destination}')
print('='*60)
total_flight = None
if not flights.empty:
    cheapest   = int(flights['Price'].min())
    total_flight = cheapest * travelers
    best        = flights.loc[flights['Price'].idxmin()]
    print(f'  ✅ Cheapest : Rs.{cheapest}/person ({best["Airline"]} | {best["Total_Stops"]})')
    print(f'  Total for {travelers} pax: Rs.{total_flight:,} (Dataset)')
    print('\n  Top 3 options:')
    for _, r in flights.nsmallest(3,'Price').iterrows():
        print(f'    ✈ {r["Airline"]:22s} {r["Total_Stops"]:15s} Rs.{r["Price"]:,}')
else:
    print(f'  ⚠️  No direct data. Book online:')
    print(f'  🔗 Google Flights : https://www.google.com/travel/flights')
    print(f'  🔗 MakeMyTrip     : https://www.makemytrip.com/flights/')
    print(f'  🔗 Skyscanner     : https://www.skyscanner.co.in/')

print('\n  🚌 Other options:')
print('  🔗 IRCTC: https://www.irctc.co.in/nget/train-search')
print('  🔗 RedBus: https://www.redbus.in/')


✈️  FLIGHTS — Mumbai → Mysore
  ⚠️  No direct data. Book online:
  🔗 Google Flights : https://www.google.com/travel/flights
  🔗 MakeMyTrip     : https://www.makemytrip.com/flights/
  🔗 Skyscanner     : https://www.skyscanner.co.in/

  🚌 Other options:
  🔗 IRCTC: https://www.irctc.co.in/nget/train-search
  🔗 RedBus: https://www.redbus.in/


In [ ]:
# ══ CELL 12: Hotels ═══════════════════════════════════
print(f'\n🏨  HOTELS — {destination}')
print('='*60)
hotel_cost_val = None
if not hotels.empty:
    top_h = hotels.sort_values('Rating', ascending=False, na_position='last').head(3)
    for _, row in top_h.iterrows():
        nm    = str(row['hotel_name'])
        price = row['Price']
        rat   = row['Rating']
        total = round(price * days * travelers, 0)
        star  = f'⭐ {rat:.1f}' if pd.notna(rat) else ''
        print(f'\n  🏨 {nm}  {star}')
        print(f'     Rs.{price:.0f}/night/person | {days}n×{travelers}pax = Rs.{total:.0f}  [✅ Dataset]')
        print(f'     🔗 {gmaps(nm, destination)}')
    hotel_cost_val = round(hotels['Price'].mean() * days * travelers, 2)
else:
    print('  ⚠️  No dataset data. Book via:')
    print(f'  🔗 Booking.com : https://www.booking.com/search.html?ss={enc(destination)}')
    print(f'  🔗 MakeMyTrip  : https://www.makemytrip.com/hotels/')
    print(f'  🔗 OYO         : https://www.oyorooms.com/')


🏨  HOTELS — Mysore

  🏨 Aishwarya Residency  ⭐ 5.0
     Rs.1000/night/person | 3n×2pax = Rs.6000  [✅ Dataset]
     🔗 https://www.google.com/maps/search/?api=1&query=Aishwarya%20Residency%20Mysore%20India

  🏨 Aishwarya Le Royal  ⭐ 5.0
     Rs.756/night/person | 3n×2pax = Rs.4536  [✅ Dataset]
     🔗 https://www.google.com/maps/search/?api=1&query=Aishwarya%20Le%20Royal%20Mysore%20India

  🏨 NI Ambaari Suites  ⭐ 5.0
     Rs.1410/night/person | 3n×2pax = Rs.8460  [✅ Dataset]
     🔗 https://www.google.com/maps/search/?api=1&query=NI%20Ambaari%20Suites%20Mysore%20India


In [ ]:
# ── UPDATED CELL 12: Budget-Aware & Multi-Traveler Itinerary ─────────────────
print('\n' + '='*60)
print('📅  GENERATING BUDGET-ALIGNED ITINERARY...')
print('='*60)

# 1. Pre-calculate Budget Tiers for the LLM
total_days = max(1, days)
budget_per_day_total = budget / total_days
budget_per_pax_per_day = budget_per_day_total / travelers

# Determine Budget Class
if budget_per_pax_per_day < 2000:
    budget_tier = "Budget/Backpacker (Focus on free sights, street food, and public transport)"
elif budget_per_pax_per_day < 7000:
    budget_tier = "Mid-Range (Mix of paid attractions, cafes, and comfortable transport)"
else:
    budget_tier = "Luxury/Premium (High-end dining, private tours, and premium experiences)"

# 2. Prepare Restaurant List
restaurant_list = (
    list(foods['name'].head(15)) if not foods.empty
    else [f'Top-rated {budget_tier} restaurants in {destination}']
)

# 3. Dynamically generate traveler outfit rules and format
traveler_outfit_rules = ""
traveler_outfit_format = ""
for i, profile in enumerate(traveler_profiles):
    traveler_outfit_rules += f"  - Traveler {i+1}: Age {profile['age']}, Gender {profile['gender']}, Style {profile['style']}.\n"
    traveler_outfit_format += f"- Traveler {i+1} ({profile['gender']}, {profile['age']}, {profile['style']}): Top: [Item] | Bottom: [Item] | Shoes: [Item]\n"

# 4. Enhanced Prompt
itinerary_prompt = f"""<task>Generate a professional travel itinerary for {travelers} travelers.</task>

<rules>
- START EXACTLY WITH "DAY 1". No preamble.
- BUDGET ALIGNMENT: The total budget is Rs.{budget}. You MUST scale the luxury of suggestions to fit this.
  - Since the daily budget is ~Rs.{int(budget_per_day_total)}, suggest {budget_tier}.
  - If budget is HIGH, suggest fine dining and premium activities from the data.
  - If budget is LOW, prioritize free landmarks and affordable local eats.
- OUTFITS: Provide a specific outfit for EACH of the {travelers} traveler(s) based on:
{traveler_outfit_rules}
  - Use live weather: {weather['temp_c']}C, {weather['desc']}.
- DATA USE: Prioritize the 'Places' and 'Restaurants' lists provided. Only fallback to general knowledge if lists are exhausted.
- Label cost sources: (Dataset) or (Estimated).
- ALL costs shown must be the TOTAL for {travelers} pax.
</rules>

<format>
DAY N - DD-MM-YYYY
--------------------------------------------------
09:00 AM | Visit    : PLACE_NAME      | Rs.COST ({travelers} pax) | (Source)
01:00 PM | Lunch    : RESTAURANT      | Rs.COST ({travelers} pax) | (Source)
03:00 PM | Visit    : PLACE_NAME      | Rs.COST ({travelers} pax) | (Source)
07:00 PM | Dinner   : RESTAURANT      | Rs.COST ({travelers} pax) | (Source)

OUTFITS FOR {travelers} TRAVELERS:
{traveler_outfit_format}--------------------------------------------------
DAY TOTAL: Rs.COST
</format>

<data>
Places: {clean_place_names}
Restaurants: {restaurant_list}
Budget Tier: {budget_tier}
Dates: {start_date} to {end_date}
Weather: {weather['desc']}, {weather['temp_c']}C
</data>"""

try:
    res = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': 'You are an expert travel planner. You align every suggestion to the user\'s specific budget tier and weather conditions.'},
            {'role': 'user', 'content': itinerary_prompt}
        ], temperature=0.2) # Slight temperature for better outfit variety

    itinerary_text = res.choices[0].message.content.strip()
    print(itinerary_text)
except Exception as e:
    print(f'Itinerary error: {e}')


📅  GENERATING BUDGET-ALIGNED ITINERARY...
DAY 1 - 04-04-2026
--------------------------------------------------
09:00 AM | Visit    : Narasimharaja Wodeyar | Rs.500 (2 pax) | (Dataset)
01:00 PM | Lunch    : The Noodle Theory | Rs.1500 (2 pax) | (Dataset)
03:00 PM | Visit    : Krishnaraja Wodeyar | Rs.300 (2 pax) | (Dataset)
07:00 PM | Dinner   : MALHAR PURE VEG | Rs.1200 (2 pax) | (Dataset)

OUTFITS FOR 2 TRAVELERS:
- Traveler 1 (Male, 21, Casual): Top: White linen shirt | Bottom: Dark blue jeans | Shoes: Black sneakers
- Traveler 2 (Female, 21, Traditional): Top: Pink silk blouse | Bottom: Navy blue salwar | Shoes: Beige juttis

--------------------------------------------------
DAY TOTAL: Rs.3300

DAY 2 - 05-04-2026
--------------------------------------------------
09:00 AM | Visit    : Private Tour of Mysore Palace | Rs.8000 (2 pax) | (Estimated)
01:00 PM | Lunch    : Mayura Dine In | Rs.1500 (2 pax) | (Dataset)
03:00 PM | Visit    : Lakshmi | Rs.200 (2 pax) | (Dataset)
07:00 PM |

In [ ]:
# ══ CELL 13: Budget Planner (LLM, single clean table) ═
print('\n💰  BUDGET PLANNER')
print('='*60)

flight_info = (
    f'Rs.{flights["Price"].min()*travelers:,.0f} ({flights["Airline"].iloc[0]}, cheapest, Dataset)'
    if not flights.empty
    else f'Not in dataset — estimate for {departure} to {destination} x{travelers} pax'
)
hotel_info  = (
    f'Rs.{hotel_cost_val:,.0f} ({days} nights x{travelers} pax, avg, Dataset)'
    if hotel_cost_val
    else f'Not in dataset — estimate for {destination} x{travelers} pax x{days} nights'
)
food_info   = (
    f'Rs.{food_cost_val:,.0f} ({days} days x{travelers} pax x3 meals, avg, Dataset)'
    if food_cost_val
    else f'Not in dataset — estimate for {destination} x{travelers} pax x{days} days x3 meals'
)

budget_prompt = f"""You are a travel budget expert for India.
Output EXACTLY ONE clean markdown table. First character is |. No text before or after.

RULES:
- All costs = TOTAL for {travelers} traveler(s)
- Dataset value → use exactly as given, mark Source as 'Dataset'
- Missing value → estimate realistic 2024 Indian rate, mark Source as 'Estimated'
- ONE table only. No duplicate tables. No prose.
- If grand total > Rs.{budget:.0f}, add a final row: | OVER BUDGET | - | -Rs.OVERAGE | |

| # | Category | Details | Cost (Rs.) | Source |
|---|----------|---------|------------|--------|
| 1 | Flights | {departure}→{destination} x{travelers} pax | FILL | FILL |
| 2 | Hotels | {days} nights x{travelers} pax | FILL | FILL |
| 3 | Food | {days} days x{travelers} pax x3 meals | FILL | FILL |
| 4 | Attractions | {preference} activities x{travelers} pax | FILL | Estimated |
| 5 | Local Transport | {days} days x{travelers} pax | FILL | Estimated |
| 6 | Shopping/Misc | {days} days x{travelers} pax | FILL | Estimated |
| TOTAL | - | For {travelers} traveler(s) | GRAND_SUM | - |

DATA:
Flights : {flight_info}
Hotels  : {hotel_info}
Food    : {food_info}
Budget  : Rs.{budget:.0f} | Travelers: {travelers} | Days: {days} | Preference: {preference}"""

budget_table = 'Budget unavailable.'
try:
    res = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role':'system',
             'content':'Output ONLY one markdown table. First char must be |. Zero prose. Zero duplicate tables.'},
            {'role':'user','content':budget_prompt}
        ], temperature=0, max_tokens=800)
    raw   = res.choices[0].message.content.strip()
    lines = [l for l in raw.split('\n') if l.strip().startswith('|')]
    budget_table = '\n'.join(lines) if lines else raw
    print(budget_table)
except Exception as e:
    print(f'❌ Budget error: {e}')


💰  BUDGET PLANNER
| # | Category | Details | Cost (Rs.) | Source |
|---|----------|---------|------------|--------|
| 1 | Flights | Mumbai→Mysore x2 pax | 8,000 | Estimated |
| 2 | Hotels | 3 nights x2 pax | 12,217 | Dataset |
| 3 | Food | 3 days x2 pax x3 meals | 5,411 | Dataset |
| 4 | Attractions | cultural activities x2 pax | 1,500 | Estimated |
| 5 | Local Transport | 3 days x2 pax | 800 | Estimated |
| 6 | Shopping/Misc | 3 days x2 pax | 1,200 | Estimated |
| TOTAL | - | For 2 traveler(s) | 29,328 | - |


In [ ]:
# ── FINAL CELL: CONSOLIDATED BURGUNDY UI & PDF GENERATION ─────────────
import gradio as gr
import pandas as pd
import os, ast, re, requests, urllib.parse, warnings
from fpdf import FPDF
from fpdf.enums import XPos, YPos
from groq import Groq
from datetime import datetime

warnings.filterwarnings('ignore')

groq_client = Groq(api_key=GROQ_API_KEY)

# --- GLOBAL HELPER FUNCTIONS ---
def enc(q):   return urllib.parse.quote(str(q))
def cln(s):   return str(s).strip().lower()
def gmaps(place, city=''):
    q = (place + ' ' + city + ' India').strip()
    return f'https://www.google.com/maps/search/?api=1&query={enc(q)}'

def get_weather(city):
    try:
        r = requests.get(f'https://wttr.in/{enc(city)}?format=j1', timeout=6).json()
        d = r['weather'][0]
        return {
            'temp_c'  : d['avgtempC'],
            'desc'    : d['hourly'][4]['weatherDesc'][0]['value'],
            'humidity': d['hourly'][4]['humidity'],
            'wind'    : d['hourly'][4]['windspeedKmph']
        }
    except:
        return {'temp_c':'N/A','desc':'N/A','humidity':'N/A','wind':'N/A'}

def wiki_snippet(name):
    try:
        r = requests.get(
            f'https://en.wikipedia.org/api/rest_v1/page/summary/{enc(name.split("(")[0])}',
            timeout=4).json()
        return r.get('extract','')[:180]
    except:
        return ''

JUNK = ['community','network','node','boundary','residential',
        'administrative','unknown','unnamed','possible']

def is_valid(name):
    return (isinstance(name, str) and len(name) > 3
            and not any(j in name.lower() for j in JUNK))

def get_places_otm(city, preference_type='cultural', limit=12):
    pref_kinds = {
        'cultural'  : 'cultural,historic,architecture,religion,museums',
        'adventure' : 'natural,sport,hiking,water,amusements',
        'foodie'    : 'foods,restaurants,markets',
        'relaxation': 'natural,parks,spas,beaches,gardens'
    }
    kinds = pref_kinds.get(preference_type, 'interesting_places,cultural,historic')
    try:
        geo = requests.get(
            f'https://api.opentripmap.com/0.1/en/places/geoname'
            f'?name={enc(city)}&country=IN&apikey={OPENTRIPMAP_API_KEY}',
            timeout=6).json()
        lat, lon = geo.get('lat'), geo.get('lon')
        if not lat or not lon:
            return [], None, None
        raw = requests.get(
            f'https://api.opentripmap.com/0.1/en/places/radius'
            f'?radius=20000&lon={lon}&lat={lat}'
            f'&kinds={kinds}&rate=2&format=json&limit=30'
            f'&apikey={OPENTRIPMAP_API_KEY}',
            timeout=8).json()
        places_otm, seen = [], set()
        for r in raw:
            nm = r.get('name','').strip()
            if nm and nm not in seen and is_valid(nm):
                seen.add(nm)
                places_otm.append({
                    'name'  : nm,
                    'gmaps' : gmaps(nm, city),
                    'wiki'  : wiki_snippet(nm)
                })
        return places_otm[:limit], lat, lon
    except Exception as e:
        return [], None, None

def get_places_llm(city, preference_type='cultural'):
    try:
        res = groq_client.chat.completions.create(
            model='llama3-70b-8192',
            messages=[
              {'role':'system',
               'content':(
                   'You are a travel data assistant. '
                   'Output ONLY a valid Python list of strings. '
                   'List ONLY real, well-known, verifiable places. '
                   'NEVER invent place names. '
                   'Format: ["Place1","Place2",...] -- nothing else.')},
              {'role':'user',
               'content':(
                   f'List exactly 10 real famous tourist attractions in {city}, India '
                   f'that are best for a {preference_type} traveller. '
                   f'Only well-known verifiable landmarks. '
                   f'Output ONLY: ["Name1","Name2","Name3","Name4","Name5",'
                   f'"Name6","Name7","Name8","Name9","Name10"]')}
            ], temperature=0)
        raw  = res.choices[0].message.content.strip()
        lst  = ast.literal_eval(raw[raw.find('['):raw.rfind(']')+1])
        return [{'name':n,'gmaps':gmaps(n,city),'wiki':wiki_snippet(n)}
                for n in lst if is_valid(n)]
    except:
        return []

def get_restaurants_llm(city, preference_type='cultural', n=8):
    pref_food = {
        'foodie'    : 'famous local delicacies, street food, must-try restaurants',
        'cultural'  : 'traditional authentic local cuisine restaurants',
        'adventure' : 'casual cafes, dhaba-style, energizing food spots',
        'relaxation': 'rooftop restaurants, scenic dining, peaceful cafes'
    }
    food_hint = pref_food.get(preference_type, 'popular local restaurants')
    try:
        res = groq_client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
              {'role':'system',
               'content':(
                   'You are a food travel expert. '
                   'Output ONLY a valid Python list of dicts. '
                   'List ONLY real, verifiable restaurant names that actually exist. '
                   'NEVER invent names. '
                   'Format exactly: [{"name":"X","cuisine":"Y","specialty":"Z"},...] -- nothing else.')},
              {'role':'user',
               'content':(
                   f'List {n} real well-known restaurants in {city}, India '
                   f'focusing on {food_hint}. '
                   f'Only places that genuinely exist. '
                   f'Output ONLY the Python list.')}
            ], temperature=0)
        raw = res.choices[0].message.content.strip()
        lst = ast.literal_eval(raw[raw.find('['):raw.rfind(']')+1])
        return [{
            'name'     : r.get('name',''),
            'cuisine'  : r.get('cuisine',''),
            'specialty': r.get('specialty',''),
            'gmaps'    : gmaps(r.get('name',''), city),
            'source'   : 'LLM Suggested'
        } for r in lst if r.get('name','').strip()]
    except:
        return []
# --- END GLOBAL HELPER FUNCTIONS ---

# ── PDF Color palette ─────────────────────────────────────────────────
C = {
    'navy'   : (15,  30,  80),
    'teal'   : (0,  128, 128),
    'gold'   : (180, 140,  40),
    'coral'  : (210,  90,  70),
    'sage'   : ( 90, 140, 110),
    'lavndr' : (110,  90, 160),
    'cream'  : (255, 252, 240),
    'ltgrey' : (245, 245, 248),
    'midgrey': (180, 180, 185),
    'dark'   : ( 30,  30,  35),
    'white'  : (255, 255, 255),
}

PREF_COLOR = {
    'foodie'    : C['coral'],
    'cultural'  : C['gold'],
    'adventure' : C['sage'],
    'relaxation': C['lavndr'],
}

# ── Emoji -> ASCII replacement map for PDF latin-1 compatibility ──────
# Maps every emoji used in cell outputs to a readable ASCII bracketed tag.
# This is the ONLY change needed for emoji support - no font changes.
EMOJI_MAP = {
    '✈️': '[Plane]',  '✈': '[Plane]',
    '\U0001f3c1': '[Dest]',
    '\U0001f465': '[Group]',
    '\U0001f4c5': '[Date]',
    '\U0001f4b0': '[Budget]',
    '\U0001f464': '[Person]',
    '\U0001f3af': '[Target]',
    '\U0001f324\uFE0F': '[Weather]', '\U0001f324': '[Weather]',
    '☀️': '[Sun]',    '☀': '[Sun]',
    '\U0001f5fa\uFE0F': '[Map]','\U0001f5fa': '[Map]',
    '\U0001f4cd': '[Pin]',
    '\U0001f374': '[Fork]',
    '\U0001f37d\uFE0F': '[Plate]', '\U0001f37d': '[Plate]',
    '\U0001f3e8': '[Hotel]',
    '⭐': '[Star]',
    '✅': '[OK]',
    '⚠️': '[Warn]',   '⚠': '[Warn]',
    '❌': '[Err]',
    '\U0001f9e0': '[AI]',
    '\U0001f4c4': '[PDF]',
    '\U0001f517': '[Link]',
    '\U0001f680': '[Go]',
    '\U0001f4a1': '[Tip]',
    '\U0001f4ca': '[Data]',
    '\U0001f511': '[Key]',
    '\U0001f4c2': '[Folder]',
    '₹': 'Rs.',            '₹': 'Rs.',
    '\U0001f30d': '[Globe]',
    '\U0001f6cc': '[Sleep]',
    '\U0001f6b4': '[Bike]',
    '\U0001f695': '[Taxi]',
    '\U0001f6b6': '[Walk]',
    '\U0001f31e': '[Sun]',
    '\U0001f308': '[Rainbow]',
    '\U0001f4b5': '[Money]',
    '\U0001f4b3': '[Card]',
    '\U0001f3a8': '[Art]',
    '\U0001f3db': '[Museum]',
    '\U0001f3d4': '[Mountain]',
    '\U0001f3d6': '[Beach]',
    '\U0001f333': '[Tree]',
    '\U0001f375': '[Tea]',
    '\U0001f376': '[Drink]',
    '\U0001f355': '[Food]',
    '\U0001f356': '[Meat]',
}

def safe(t):
    """Replace emojis with ASCII tags then strip any remaining non-latin-1 chars."""
    t = str(t).strip()
    for emoji, replacement in EMOJI_MAP.items():
        t = t.replace(emoji, replacement)
    return t.encode('latin-1', 'replace').decode('latin-1')

class PDF(FPDF):
    def __init__(self, orientation='P', unit='mm', format='A4',
                 destination_city="Unknown", accent_color=None):
        super().__init__(orientation, unit, format)
        self.destination_city = destination_city
        self.accent_color = accent_color or C['teal']

    def header(self):
        pass

    def footer(self):
        self.set_y(-13)
        self.set_font('Helvetica', '', 8)
        self.set_text_color(*C['midgrey'])
        self.cell(0, 8,
                  safe(f'Travel Planner PRO  |  {self.destination_city}  |  Page {self.page_no()}'),
                  align='C')

def filled_rect(pdf_obj, x, y, w, h, rgb):
    pdf_obj.set_fill_color(*rgb)
    pdf_obj.rect(x, y, w, h, 'F')

def section_header(pdf_obj, title, icon='', color=None):
    color = color or pdf_obj.accent_color
    pdf_obj.ln(3)
    filled_rect(pdf_obj, 10, pdf_obj.get_y(), 190, 9, color)
    pdf_obj.set_font('Helvetica', 'B', 11)
    pdf_obj.set_text_color(*C['white'])
    pdf_obj.set_xy(13, pdf_obj.get_y() + 1.5)
    pdf_obj.cell(0, 6, safe(f'{icon}  {title}'),
                 new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf_obj.set_text_color(*C['dark'])
    pdf_obj.ln(2)

def info_card(pdf_obj, label, value, icon='', accent_color=None):
    accent_color = accent_color or pdf_obj.accent_color
    filled_rect(pdf_obj, 10, pdf_obj.get_y(), 190, 7, C['ltgrey'])
    pdf_obj.set_font('Helvetica', 'B', 9)
    pdf_obj.set_text_color(*accent_color)
    pdf_obj.set_x(13)
    pdf_obj.cell(45, 7, safe(f'{icon} {label}'), ln=False)
    pdf_obj.set_font('Helvetica', '', 9)
    pdf_obj.set_text_color(*C['dark'])
    pdf_obj.cell(0, 7, safe(str(value)),
                 new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf_obj.ln(1)

def divider(pdf_obj):
    pdf_obj.set_draw_color(*C['midgrey'])
    pdf_obj.set_line_width(0.3)
    pdf_obj.line(10, pdf_obj.get_y(), 200, pdf_obj.get_y())
    pdf_obj.ln(2)


# ── AUTO-FETCH: pull all results from previous cells ──────────────────
def _auto_fetch_previous_results():
    """
    Reads all variables populated by earlier cells (cells 6-14) so the UI
    and PDF always reflect exactly what was computed and displayed there.
    Returns a dict with every piece of state the final cell needs.
    Priority order: earlier-cell computed values -> sensible defaults.
    """
    g = globals()
    return {
        # Trip basics (Cell 6)
        'dep'             : g.get('departure',   ''),
        'dest'            : g.get('destination', ''),
        'travelers'       : int(g.get('travelers', 1)),
        's_date'          : g.get('start_date',  ''),
        'e_date'          : g.get('end_date',    ''),
        'bud'             : float(g.get('budget', 0)),
        'pref'            : g.get('preference',  'cultural'),
        'days'            : int(g.get('days', 1)),
        'profiles'        : g.get('traveler_profiles', []),
        # Filtered datasets (Cell 7)
        'hotels'          : g.get('hotels',  pd.DataFrame()),
        'foods'           : g.get('foods',   pd.DataFrame()),
        'flights'         : g.get('flights', pd.DataFrame()),
        # Weather (Cell 8)
        'weather'         : g.get('weather',
                                  {'temp_c':'N/A','desc':'N/A','humidity':'N/A','wind':'N/A'}),
        # Places (Cell 9)
        'places'          : g.get('places', []),
        'clean_place_names': g.get('clean_place_names', []),
        # Restaurants (Cell 10)
        'restaurant_data' : g.get('restaurant_data', []),
        'restaurant_list' : g.get('restaurant_list', []),
        'food_cost_val'   : g.get('food_cost_val',   None),
        # Flights total (Cell 11)
        'total_flight'    : g.get('total_flight', None),
        # Hotels cost (Cell 12)
        'hotel_cost_val'  : g.get('hotel_cost_val', None),
        # Itinerary text (Cell 13)
        'itinerary_text'  : g.get('itinerary_text', ''),
        # Budget table markdown (Cell 14)
        'budget_table'    : g.get('budget_table', ''),
    }


# ── Main Logic Wrapper ────────────────────────────────────────────────
def run_full_travel_planner(dep, dest, pax, s_date, e_date, bud, prefs_list, t_data):
    yield "Initializing Planner...", None

    global GROQ_API_KEY, OPENTRIPMAP_API_KEY
    global swiggy_df, hotel_df, flight_df

    try:
        # Pull everything computed in previous cells first
        prev = _auto_fetch_previous_results()

        # Resolve each field: prefer UI input if non-empty/non-zero,
        # otherwise fall back to what earlier cells already computed.
        departure_city   = str(dep).strip()    if str(dep).strip()    else prev['dep']
        destination_city = str(dest).strip()   if str(dest).strip()   else prev['dest']
        num_travelers    = int(pax)             if pax                 else prev['travelers']
        start_date_str   = str(s_date).strip()  if str(s_date).strip() else prev['s_date']
        end_date_str     = str(e_date).strip()  if str(e_date).strip() else prev['e_date']
        total_budget     = float(bud)           if bud                 else prev['bud']

        pref_map = {
            'Foodie':'foodie','Cultural':'cultural',
            'Adventure':'adventure','Relaxation':'relaxation'
        }
        preference = pref_map.get(prefs_list[0], 'cultural') if prefs_list else prev['pref']
        current_accent_color = PREF_COLOR.get(preference, C['teal'])

        # Date / days
        def parse_date(d):
            for fmt in ('%Y-%m-%d', '%d-%m-%Y', '%d/%m/%Y'):
                try:
                    return datetime.strptime(str(d), fmt).date()
                except:
                    pass
            return datetime.today().date()

        sd_obj = parse_date(start_date_str)
        ed_obj = parse_date(end_date_str)
        days   = max((ed_obj - sd_obj).days + 1, prev.get('days', 1))

        # Traveler profiles: prefer UI table, fall back to cell 6 profiles
        traveler_profiles_list = []
        try:
            for row in t_data.values.tolist():
                if str(row[0]).strip() not in ('', 'nan'):
                    traveler_profiles_list.append({
                        'age'   : str(row[0]).strip(),
                        'gender': str(row[1]).strip(),
                        'style' : str(row[2]).strip()

                    })
        except Exception:
            pass
        if not traveler_profiles_list:
            traveler_profiles_list = (
                prev['profiles'] or [{'age':'N/A','gender':'N/A','style':'N/A'}]
            )

        dest_same = cln(prev['dest']) == cln(destination_city)

        # Weather: reuse cell 8 result if destination matches, else re-fetch
        if dest_same and prev['weather'] and prev['weather'].get('desc') not in (None,'N/A',''):
            weather = prev['weather']
        else:
            yield f"Fetching weather for {destination_city}...", None
            weather = get_weather(destination_city)

        # Datasets: reuse cell 7 results if destination matches
        yield "Filtering datasets...", None
        if dest_same and not prev['hotels'].empty:
            hotels  = prev['hotels']
            foods   = prev['foods']
            flights = prev['flights']
        else:
            hotels  = hotel_df[
                hotel_df['City'].apply(cln).str.contains(cln(destination_city), na=False)
            ].copy()
            foods   = swiggy_df[
                swiggy_df['city_clean'].apply(cln).str.contains(cln(destination_city), na=False)
            ].copy()
            flights = flight_df[
                flight_df['Source'].apply(cln).str.contains(cln(departure_city)) &
                flight_df['Destination'].apply(cln).str.contains(cln(destination_city))
            ].copy()

        # Places: reuse cell 9 results if available and destination matches
        if dest_same and prev['places']:
            places            = prev['places']
            clean_place_names = prev['clean_place_names']
        else:
            yield "Fetching attractions...", None
            places, _, __ = get_places_otm(destination_city, preference)
            if len(places) < 3:
                places = get_places_llm(destination_city, preference)
            clean_place_names = [p['name'] for p in places if is_valid(p['name'])]
        if not clean_place_names:
            clean_place_names = [f'{destination_city} city centre']

        # Restaurants: reuse cell 10 results if available and destination matches
        if dest_same and prev['restaurant_data']:
            restaurant_data = prev['restaurant_data']
            food_cost_val   = prev['food_cost_val']
        else:
            yield "Finding restaurants...", None
            restaurant_data = []
            food_cost_val   = None
            if not foods.empty:
                top_foods = foods.sort_values(
                    'rating_clean', ascending=False, na_position='last').head(8)
                for _, row in top_foods.iterrows():
                    restaurant_data.append({
                        'name'   : str(row['name']),
                        'cuisine': str(row.get('cuisine','')) if pd.notna(row.get('cuisine')) else '',
                        'cost'   : row['cost_numeric'],
                        'rating' : row['rating_clean'],
                        'gmaps'  : gmaps(str(row['name']), destination_city),
                        'source' : 'Dataset'
                    })
                avg_cost = foods['cost_numeric'].dropna()
                if not avg_cost.empty:
                    food_cost_val = round(avg_cost.mean() * 3 * days * num_travelers, 2)
            else:
                for r in get_restaurants_llm(destination_city, preference):
                    restaurant_data.append({
                        'name'     : r['name'],
                        'cuisine'  : r['cuisine'],
                        'specialty': r.get('specialty',''),
                        'gmaps'    : gmaps(r.get('name',''), destination_city),
                        'source'   : 'LLM Suggested'
                    })
        restaurant_list = [r['name'] for r in restaurant_data]

        # Itinerary text: reuse cell 13 result if destination matches
        if dest_same and prev['itinerary_text']:
            itinerary_text = prev['itinerary_text']
        else:
            yield "Generating itinerary...", None
            budget_per_day   = total_budget / max(1, days)
            budget_per_pax   = budget_per_day / num_travelers
            if budget_per_pax < 2000:
                budget_tier = "Budget/Backpacker (Focus on free sights, street food, and public transport)"
            elif budget_per_pax < 7000:
                budget_tier = "Mid-Range (Mix of paid attractions, cafes, and comfortable transport)"
            else:
                budget_tier = "Luxury/Premium (High-end dining, private tours, and premium experiences)"

            llm_restaurant_list = (
                list(foods['name'].head(15)) if not foods.empty
                else [f'Top-rated {budget_tier} restaurants in {destination_city}']
            )
            traveler_info_llm = "".join(
                f"- Traveler {i+1} (Age {p['age']}, {p['gender']}, Style: {p['style']})\n"
                for i, p in enumerate(traveler_profiles_list)
            )
            outfit_example = (
                f"- Traveler 1 ({traveler_profiles_list[0]['gender']},"
                f" {traveler_profiles_list[0]['age']}): Top: [Item] | Bottom: [Item] | Shoes: [Item]"
            )
            time_example = f"HH:MM AM/PM | Activity : NAME | Rs.COST ({num_travelers} pax) | (Source)"

            itinerary_prompt = (
                f"<task>Generate a professional travel itinerary for {num_travelers} travelers.</task>\n\n"
                "<rules>\n"
                "- START EXACTLY WITH \"DAY 1\". No preamble.\n"
                f"- BUDGET ALIGNMENT: Total budget Rs.{total_budget}. Suggest {budget_tier}.\n"
                f"- OUTFITS: Specific outfit for EACH traveler:\n{traveler_info_llm.strip()}\n"
                f"- Use live weather: {weather['temp_c']}C, {weather['desc']}.\n"
                "- Prioritize Places and Restaurants lists provided.\n"
                "- Label cost sources: (Dataset) or (Estimated).\n"
                f"- ALL costs shown must be TOTAL for {num_travelers} pax.\n"
                "</rules>\n\n"
                "<format>\n"
                f"DAY N - {start_date_str}\n"
                "--------------------------------------------------\n"
                f"{time_example}\n\n"
                f"OUTFITS FOR {num_travelers} TRAVELERS:\n"
                f"{outfit_example}\n"
                "--------------------------------------------------\n"
                "DAY TOTAL: Rs.COST\n"
                "</format>\n\n"
                "<data>\n"
                f"Places: {clean_place_names}\n"
                f"Restaurants: {llm_restaurant_list}\n"
                f"Budget Tier: {budget_tier}\n"
                f"Dates: {start_date_str} to {end_date_str}\n"
                f"Weather: {weather['desc']}, {weather['temp_c']}C\n"
                "</data>"
            )

            res = groq_client.chat.completions.create(
                model='llama-3.1-8b-instant',
                messages=[
                    {'role':'system',
                     'content':'You are an expert travel planner aligning suggestions to budget and weather.'},
                    {'role':'user','content':itinerary_prompt}
                ], temperature=0.2)
            itinerary_text = res.choices[0].message.content.strip()

        # Budget table: reuse cell 14 result if destination matches
        if dest_same and prev['budget_table']:
            budget_table = prev['budget_table']
        else:
            yield "Calculating budget...", None
            flight_info_str = (
                f'Rs.{flights["Price"].min()*num_travelers:,.0f}'
                f' ({flights["Airline"].iloc[0]}, cheapest, Dataset)'
                if not flights.empty
                else f'Not in dataset -- estimate {departure_city} to {destination_city} x{num_travelers} pax'
            )
            hotel_cost_val_local = (
                round(hotels['Price'].mean() * days * num_travelers, 2)
                if not hotels.empty else None
            )
            hotel_info_str = (
                f'Rs.{hotel_cost_val_local:,.0f} ({days} nights x{num_travelers} pax, avg, Dataset)'
                if hotel_cost_val_local
                else f'Not in dataset -- estimate {destination_city} x{num_travelers} pax x{days} nights'
            )
            food_info_str = (
                f'Rs.{food_cost_val:,.0f} ({days} days x{num_travelers} pax x3 meals, avg, Dataset)'
                if food_cost_val
                else f'Not in dataset -- estimate {destination_city} x{num_travelers} pax x{days} days x3 meals'
            )

            budget_prompt = (
                "You are a travel budget expert for India.\n"
                "Output EXACTLY ONE clean markdown table. First character is |. No text before or after.\n\n"
                "RULES:\n"
                f"- All costs = TOTAL for {num_travelers} traveler(s)\n"
                "- Dataset value use exactly as given, mark Source as 'Dataset'\n"
                "- Missing value estimate realistic 2024 Indian rate, mark Source as 'Estimated'\n"
                "- ONE table only. No duplicate tables. No prose.\n"
                f"- If grand total > Rs.{total_budget:.0f}, add row: | OVER BUDGET | - | -Rs.OVERAGE | |\n\n"
                "| # | Category | Details | Cost (Rs.) | Source |\n"
                "|---|----------|---------|------------|--------|\n"
                f"| 1 | Flights | {departure_city}->{destination_city} x{num_travelers} pax | FILL | FILL |\n"
                f"| 2 | Hotels | {days} nights x{num_travelers} pax | FILL | FILL |\n"
                f"| 3 | Food | {days} days x{num_travelers} pax x3 meals | FILL | FILL |\n"
                f"| 4 | Attractions | {preference} x{num_travelers} pax | FILL | Estimated |\n"
                f"| 5 | Local Transport | {days} days x{num_travelers} pax | FILL | Estimated |\n"
                f"| 6 | Shopping/Misc | {days} days x{num_travelers} pax | FILL | Estimated |\n"
                f"| TOTAL | - | For {num_travelers} traveler(s) | GRAND_SUM | - |\n\n"
                "DATA:\n"
                f"Flights : {flight_info_str}\n"
                f"Hotels  : {hotel_info_str}\n"
                f"Food    : {food_info_str}\n"
                f"Budget  : Rs.{total_budget:.0f} | Travelers: {num_travelers}"
                f" | Days: {days} | Preference: {preference}"
            )

            res = groq_client.chat.completions.create(
                model='llama-3.1-8b-instant',
                messages=[
                    {'role':'system',
                     'content':'Output ONLY one markdown table. First char must be |. Zero prose.'},
                    {'role':'user','content':budget_prompt}
                ], temperature=0, max_tokens=800)
            raw   = res.choices[0].message.content.strip()
            lines = [l for l in raw.split('\n') if l.strip().startswith('|')]
            budget_table = '\n'.join(lines) if lines else raw

        # Derive totals for PDF
        flight_cost_total = (
            int(flights['Price'].min() * num_travelers) if not flights.empty else None
        )
        hotel_cost_val_pdf = (
            round(hotels['Price'].mean() * days * num_travelers, 2)
            if not hotels.empty else None
        )

        # ── BUILD PDF ─────────────────────────────────────────────────
        yield "Compiling PDF Document...", None

        pdf = PDF(destination_city=destination_city, accent_color=current_accent_color)
        pdf.set_auto_page_break(True, margin=18)

        # ===== PAGE 1: COVER =====
        pdf.add_page()
        filled_rect(pdf, 0, 0, 210, 60, C['navy'])
        pdf.set_font('Helvetica', 'B', 24)
        pdf.set_text_color(*C['white'])
        pdf.set_xy(0, 12)
        pdf.cell(210, 12, safe('TRAVEL PLANNER'), align='C',
                 new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font('Helvetica', '', 13)
        pdf.set_text_color(*C['gold'])
        pdf.cell(210, 8, safe(f'{departure_city}  ->  {destination_city}'),
                 align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font('Helvetica', '', 10)
        pdf.set_text_color(*C['cream'])
        pdf.cell(210, 7,
                 safe(f'{start_date_str}  to  {end_date_str}'
                      f'   |   {num_travelers} Traveler(s)'
                      f'   |   Budget Rs.{total_budget:,.0f}'),
                 align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        filled_rect(pdf, 70, 52, 70, 12, current_accent_color)
        pdf.set_font('Helvetica', 'B', 10)
        pdf.set_text_color(*C['white'])
        pdf.set_xy(70, 54)
        pdf.cell(70, 8, safe(f'Trip Style: {preference.upper()}'), align='C')
        pdf.ln(18)

        section_header(pdf, 'TRIP OVERVIEW', icon='[Globe]', color=C['navy'])
        info_card(pdf, 'From',       departure_city,   '')
        info_card(pdf, 'To',         destination_city, '')
        info_card(pdf, 'Dates',      f'{start_date_str} to {end_date_str} ({days} days)', '')
        info_card(pdf, 'Travelers',  str(num_travelers), '')
        info_card(pdf, 'Budget',     f'Rs.{total_budget:,.0f}', '')
        info_card(pdf, 'Preference', preference.title(), '')
        pdf.ln(3)

        section_header(pdf, 'TRAVELER PROFILES', icon='[Person]', color=C['teal'])
        for i, p in enumerate(traveler_profiles_list):
            info_card(pdf, f'Traveler {i+1}',
                      f"Age {p['age']}  |  {p['gender']}  |  Style: {p['style']}", '')
        pdf.ln(3)

        section_header(pdf, f'LIVE WEATHER -- {destination_city.upper()}',
                       icon='[Weather]', color=C['teal'])
        info_card(pdf, 'Conditions',  safe(str(weather['desc'])),   '')
        info_card(pdf, 'Temperature', f"{weather['temp_c']} C",     '')
        info_card(pdf, 'Humidity',    f"{weather['humidity']}%",    '')
        info_card(pdf, 'Wind',        f"{weather['wind']} km/h",    '')
        pdf.ln(3)

        section_header(pdf, 'FLIGHTS', icon='[Plane]', color=C['navy'])
        if not flights.empty:
            for _, r in flights.nsmallest(3, 'Price').iterrows():
                info_card(pdf, safe(str(r['Airline'])),
                          f'{safe(str(r["Total_Stops"]))}  |  Rs.{int(r["Price"]):,}/person  [Dataset]',
                          '')
            if flight_cost_total:
                info_card(pdf, 'Total (cheapest)',
                          f'Rs.{flight_cost_total:,} for {num_travelers} pax', '')
        else:
            pdf.set_font('Helvetica', 'I', 9)
            pdf.set_text_color(*C['coral'])
            pdf.set_x(13)
            pdf.multi_cell(0, 6,
                safe('No flights in dataset. Book via: google.com/travel/flights | makemytrip.com'))
            pdf.set_text_color(*C['dark'])
        pdf.ln(3)

        section_header(pdf, 'HOTELS', icon='[Hotel]', color=C['gold'])
        if not hotels.empty:
            for _, row in hotels.sort_values(
                    'Rating', ascending=False, na_position='last').head(3).iterrows():
                rat = f'  Rated {row["Rating"]:.1f}' if pd.notna(row['Rating']) else ''
                info_card(pdf, safe(str(row['hotel_name'])),
                          f'Rs.{row["Price"]:.0f}/night/pax{rat}'
                          f'  |  {days}n x{num_travelers}pax'
                          f' = Rs.{row["Price"]*days*num_travelers:.0f}  [Dataset]', '')
        else:
            pdf.set_font('Helvetica', 'I', 9)
            pdf.set_text_color(*C['coral'])
            pdf.set_x(13)
            pdf.multi_cell(0, 6,
                safe('No hotel data. Book via: booking.com | makemytrip.com/hotels | oyorooms.com'))
            pdf.set_text_color(*C['dark'])
        pdf.ln(3)

        # ===== PAGE 2: PLACES & RESTAURANTS =====
        pdf.add_page()
        filled_rect(pdf, 0, 0, 210, 14, current_accent_color)
        pdf.set_font('Helvetica', 'B', 13)
        pdf.set_text_color(*C['white'])
        pdf.set_xy(0, 3)
        pdf.cell(210, 8, safe(f'PLACES & FOOD -- {destination_city.upper()}'),
                 align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_text_color(*C['dark'])
        pdf.ln(5)

        section_header(pdf, f'PLACES TO VISIT ({preference.upper()})',
                       icon='[Pin]', color=current_accent_color)
        for p in places:
            pdf.set_font('Helvetica', 'B', 10)
            pdf.set_text_color(*current_accent_color)
            pdf.set_x(13)
            pdf.cell(0, 6, safe(f'  {p["name"]}'),
                     new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            if p.get('wiki'):
                pdf.set_font('Helvetica', '', 8)
                pdf.set_text_color(*C['dark'])
                pdf.set_x(16)
                pdf.multi_cell(175, 5, safe(p['wiki'][:160]))
            pdf.set_font('Helvetica', 'I', 8)
            pdf.set_text_color(*C['teal'])
            pdf.set_x(16)
            pdf.cell(0, 5, safe(f'Maps: {p["gmaps"]}'),
                     new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            pdf.set_text_color(*C['dark'])
            pdf.ln(1)

        section_header(pdf, 'RESTAURANTS', icon='[Fork]', color=C['coral'])
        for r in restaurant_data:
            pdf.set_font('Helvetica', 'B', 10)
            pdf.set_text_color(*C['coral'])
            pdf.set_x(13)
            pdf.cell(0, 6, safe(f'  {r["name"]}  [{r["source"]}]'),
                     new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            details = []
            if r.get('cuisine'):
                details.append(f'Cuisine: {r["cuisine"]}')
            if pd.notna(r.get('rating')) and r.get('rating'):
                details.append(f'Rating: {r["rating"]:.1f}')
            if pd.notna(r.get('cost')) and r.get('cost'):
                details.append(f'~Rs.{r["cost"]:.0f}/person')
            if details:
                pdf.set_font('Helvetica', '', 8)
                pdf.set_text_color(*C['dark'])
                pdf.set_x(16)
                pdf.cell(0, 5, safe('  |  '.join(details)),
                         new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            pdf.set_font('Helvetica', 'I', 8)
            pdf.set_text_color(*C['teal'])
            pdf.set_x(16)
            pdf.cell(0, 5, safe(f'Maps: {r["gmaps"]}'),
                     new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            pdf.set_text_color(*C['dark'])
            pdf.ln(1)

        # ===== PAGE 3+: ITINERARY =====
        pdf.add_page()
        filled_rect(pdf, 0, 0, 210, 14, C['navy'])
        pdf.set_font('Helvetica', 'B', 13)
        pdf.set_text_color(*C['white'])
        pdf.set_xy(0, 3)
        pdf.cell(210, 8, safe('DAY-WISE ITINERARY + OUTFIT GUIDE'),
                 align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_text_color(*C['dark'])
        pdf.ln(5)

        day_num    = 0
        day_colors = [C['navy'], C['teal'], current_accent_color,
                      C['sage'], C['lavndr'], C['coral']]

        for line in itinerary_text.split('\n'):
            ls = line.strip()
            if not ls:
                pdf.ln(1)
                continue
            upper = ls.upper()

            # Day header detection (handles both — and -)
            if upper.startswith('DAY ') and ('--' in ls or '-' in ls or '\u2014' in ls or '\u2013' in ls):
                day_num += 1
                dc = day_colors[(day_num - 1) % len(day_colors)]
                if pdf.get_y() + 30 > pdf.h - pdf.b_margin:
                    pdf.add_page()
                    filled_rect(pdf, 0, 0, 210, 14, C['navy'])
                    pdf.set_font('Helvetica', 'B', 13)
                    pdf.set_text_color(*C['white'])
                    pdf.set_xy(0, 3)
                    pdf.cell(210, 8, safe('DAY-WISE ITINERARY + OUTFIT GUIDE (Cont.)'),
                             align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
                    pdf.set_text_color(*C['dark'])
                    pdf.ln(5)
                if day_num > 1:
                    pdf.ln(3)
                filled_rect(pdf, 10, pdf.get_y(), 190, 10, dc)
                pdf.set_font('Helvetica', 'B', 11)
                pdf.set_text_color(*C['white'])
                pdf.set_x(13)
                pdf.cell(0, 10, safe(ls),
                         new_x=XPos.LMARGIN, new_y=YPos.NEXT)
                pdf.set_text_color(*C['dark'])
                pdf.ln(2)

            elif set(ls) <= set('=-'):
                divider(pdf)

            elif ('OUTFIT' in upper
                  or ls.startswith('Traveler')
                  or ls.startswith('- Traveler')):
                pdf.set_font('Helvetica', 'B', 9)
                pdf.set_text_color(*current_accent_color)
                pdf.set_x(13)
                pdf.multi_cell(184, 6, safe(ls))
                pdf.set_text_color(*C['dark'])

            elif 'DAY' in upper and 'TOTAL' in upper:
                filled_rect(pdf, 10, pdf.get_y(), 190, 8, C['ltgrey'])
                pdf.set_font('Helvetica', 'B', 10)
                pdf.set_text_color(*C['navy'])
                pdf.set_x(13)
                pdf.cell(0, 8, safe(ls),
                         new_x=XPos.LMARGIN, new_y=YPos.NEXT)
                pdf.set_text_color(*C['dark'])
                pdf.ln(2)

            elif '|' in ls and re.match(r'\d+:\d+', ls[:6]):
                parts    = ls.split('|')
                time_act = parts[0].strip()
                rest     = ' | '.join(p.strip() for p in parts[1:])
                pdf.set_font('Helvetica', 'B', 8)
                pdf.set_text_color(*current_accent_color)
                pdf.set_x(13)
                pdf.cell(32, 6, safe(time_act), ln=False)
                pdf.set_font('Helvetica', '', 8)
                pdf.set_text_color(*C['dark'])
                pdf.multi_cell(158, 6, safe(rest))

            else:
                pdf.set_font('Helvetica', '', 9)
                pdf.set_x(13)
                pdf.multi_cell(184, 6, safe(ls))

        # ===== BUDGET PAGE =====
        pdf.add_page()
        filled_rect(pdf, 0, 0, 210, 14, C['gold'])
        pdf.set_font('Helvetica', 'B', 13)
        pdf.set_text_color(*C['navy'])
        pdf.set_xy(0, 3)
        pdf.cell(210, 8,
                 safe(f'BUDGET PLANNER -- {num_travelers} TRAVELER(S)'),
                 align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_text_color(*C['dark'])
        pdf.ln(8)

        row_alt = False
        for line in budget_table.split('\n'):
            if not line.strip().startswith('|'):
                continue
            if re.match(r'^[|\-\s]+$', line):
                continue
            cols = [c.strip() for c in line.split('|') if c.strip()]
            if not cols:
                continue

            is_header = any(c.lower() in ['#', 'category'] for c in cols)
            is_total  = any('total' in c.lower() for c in cols)
            is_over   = any('over' in c.lower() or 'budget' in c.lower() for c in cols)

            if is_header:
                filled_rect(pdf, 10, pdf.get_y(), 190, 8, C['navy'])
                pdf.set_font('Helvetica', 'B', 8)
                pdf.set_text_color(*C['white'])
            elif is_total:
                filled_rect(pdf, 10, pdf.get_y(), 190, 8, C['teal'])
                pdf.set_font('Helvetica', 'B', 9)
                pdf.set_text_color(*C['white'])
            elif is_over:
                filled_rect(pdf, 10, pdf.get_y(), 190, 8, C['coral'])
                pdf.set_font('Helvetica', 'B', 9)
                pdf.set_text_color(*C['white'])
            else:
                filled_rect(pdf, 10, pdf.get_y(), 190, 8,
                            C['ltgrey'] if row_alt else C['white'])
                pdf.set_font('Helvetica', '', 8)
                pdf.set_text_color(*C['dark'])
                row_alt = not row_alt

            col_widths = [10, 40, 70, 40, 30]
            pdf.set_x(10)
            for ci, (w, col) in enumerate(zip(col_widths, cols[:5])):
                align = 'R' if ci >= 3 else 'L'
                col   = col.replace('**', '').strip()
                pdf.cell(w, 8, safe(col[:30]), border=0, ln=False, align=align)
            pdf.ln()
            pdf.set_text_color(*C['dark'])

        pdf.ln(8)
        filled_rect(pdf, 10, pdf.get_y(), 190, 20, C['ltgrey'])
        pdf.set_font('Helvetica', 'I', 8)
        pdf.set_text_color(*C['dark'])
        pdf.set_x(13)
        pdf.multi_cell(184, 6,
            safe('Note: "Dataset" = from your uploaded files. '
                 '"Estimated" = LLM estimate for current Indian market rates (2024). '
                 'All costs are for the total group unless stated. '
                 'Restaurant Maps links open Google Maps for real-time directions.'))

        fname = f'TravelPlan_{destination_city}_{num_travelers}pax_{preference}.pdf'
        pdf.output(fname)
        yield f"Success! Itinerary for {destination_city} is ready.", fname

    except Exception as e:
        import traceback
        yield f"Error: {str(e)}\n{traceback.format_exc()}", None


# ── Auto-populate UI defaults from previous cells ─────────────────────
_prev = _auto_fetch_previous_results()

_default_dep  = _prev['dep']
_default_dest = _prev['dest']
_default_pax  = _prev['travelers']
_default_bud  = _prev['bud']

def _norm_date(d):
    """Accept DD-MM-YYYY or YYYY-MM-DD, always return YYYY-MM-DD for the UI."""
    for fmt in ('%d-%m-%Y', '%Y-%m-%d', '%d/%m/%Y'):
        try:
            return datetime.strptime(str(d), fmt).strftime('%d-%m-%Y')
        except:
            pass
    return str(d)

_default_s = _norm_date(_prev['s_date'])
_default_e = _norm_date(_prev['e_date'])

_pref_display = {
    'foodie':'Foodie','cultural':'Cultural',
    'adventure':'Adventure','relaxation':'Relaxation'
}
_default_pref = [_pref_display.get(_prev['pref'], 'Cultural')]

if _prev['profiles']:
    _default_rows = [[p['age'], p['gender'], p['style']] for p in _prev['profiles']]
else:
    _default_rows = [['', '', '']]
while len(_default_rows) < 2:
    _default_rows.append(['', '', ''])


# ── Burgundy Theme ────────────────────────────────────────────────────
burgundy_theme = gr.themes.Soft(primary_hue="rose").set(
    button_primary_background_fill="#800020",
    button_primary_text_color="white"
)

# ── UI Layout ─────────────────────────────────────────────────────────
with gr.Blocks(theme=burgundy_theme, title="Travel Planner") as demo:
    gr.Markdown("# <span style='color:#800020'>Travel Planner PRO</span>")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Trip Basics")
            dep_in  = gr.Textbox(label="Departure City",           value=_default_dep)
            dest_in = gr.Textbox(label="Destination City",         value=_default_dest)
            with gr.Row():
                pax_in = gr.Number(label="Travelers",              value=_default_pax)
                bud_in = gr.Number(label="Budget (Rs.)",           value=_default_bud)
            with gr.Row():
                s_in = gr.Textbox(label="Start Date (DD-MM-YYYY)", value=_default_s)
                e_in = gr.Textbox(label="End Date   (DD-MM-YYYY)", value=_default_e)
            prefs_in = gr.CheckboxGroup(
                ["Foodie", "Cultural", "Adventure", "Relaxation"],
                label="Preferences", value=_default_pref)

        with gr.Column():
            gr.Markdown("### Traveler Profiles")
            t_table = gr.Dataframe(
                headers=["Age", "Gender", "Style"],
                datatype=["number", "str", "str"],
                row_count=max(2, len(_default_rows)),
                col_count=(3, "fixed"),
                value=_default_rows,
                label="Enter Details for Each Traveler"
            )

    run_btn = gr.Button("GENERATE MY PERSONALISED PLAN", variant="primary")

    with gr.Row():
        status_out = gr.Textbox(label="System Status", interactive=False)
        file_out   = gr.File(label="Download PDF Itinerary")

    run_btn.click(
        fn=run_full_travel_planner,
        inputs=[dep_in, dest_in, pax_in, s_in, e_in, bud_in, prefs_in, t_table],
        outputs=[status_out, file_out]
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b87ff4d303611a6914.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b87ff4d303611a6914.gradio.live
